# Vosk

In [ ]:
!pip install -q vosk
!pip install -q jiwer hazm pandas tqdm soundfile

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 76.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 70.2 MB/s eta 0:00:00


In [ ]:
!pip install librosa

In [ ]:
import os
import re
import json
import time
import random

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import hazm

from tqdm import tqdm
from jiwer import wer, cer

from vosk import Model, KaldiRecognizer

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [ ]:
!wget https://alphacephei.com/vosk/models/vosk-model-fa-0.42.zip

!unzip -q vosk-model-fa-0.42.zip

!ls

--2026-06-19 15:12:15--  https://alphacephei.com/vosk/models/vosk-model-fa-0.42.zip
Resolving alphacephei.com (alphacephei.com)... 188.40.21.16, 2a01:4f8:13a:279f::2
Connecting to alphacephei.com (alphacephei.com)|188.40.21.16|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1667089770 (1.6G) [application/zip]
Saving to: ‘vosk-model-fa-0.42.zip’

vosk-model-fa-0.42. 100%[===================>]   1.55G  19.5MB/s    in 84s     

2026-06-19 15:13:40 (18.9 MB/s) - ‘vosk-model-fa-0.42.zip’ saved [1667089770/1667089770]

sample_data  vosk-model-fa-0.42  vosk-model-fa-0.42.zip


In [ ]:
from vosk import Model

VOSK_MODEL_PATH = "/content/vosk-model-fa-0.42"

model = Model(VOSK_MODEL_PATH)

print("Vosk model loaded successfully")

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/asr_project/asr_data/fa_test_only"

TSV_PATH = os.path.join(DATA_ROOT, "test.tsv")
AUDIO_ROOT = os.path.join(DATA_ROOT, "clips")

df = pd.read_csv(TSV_PATH, sep="\t")

df = df.dropna(
    subset=["path", "sentence"]
).reset_index(drop=True)

df["audio_path"] = df["path"].apply(
    lambda x: os.path.join(AUDIO_ROOT, x)
)

print("Total samples:", len(df))


Total samples: 10519


In [ ]:
CHECKPOINT_PATH = (
    "/content/drive/MyDrive/asr_project/"
    "vosk_checkpoint.csv"
)

if os.path.exists(CHECKPOINT_PATH):

    results_df = pd.read_csv(
        CHECKPOINT_PATH
    )

    processed = set(
        results_df["audio_path"].tolist()
    )

    print(
        "Resuming from",
        len(results_df),
        "samples"
    )

else:

    results_df = pd.DataFrame(columns=[

        "audio_path",

        "reference_raw",

        "prediction_raw",

        "audio_duration",

        "inference_time"
    ])

    processed = set()

In [ ]:
def transcribe_vosk(audio_path, model):

    audio, sr = sf.read(audio_path)

    # stereo -> mono
    if len(audio.shape) > 1:
        audio = np.mean(
            audio,
            axis=1
        )

    # resample
    if sr != 16000:

        audio = librosa.resample(
            audio,
            orig_sr=sr,
            target_sr=16000
        )

        sr = 16000

    audio_int16 = (
        audio * 32767
    ).astype(np.int16)

    rec = KaldiRecognizer(
        model,
        16000
    )

    rec.AcceptWaveform(
        audio_int16.tobytes()
    )

    result = json.loads(
        rec.FinalResult()
    )

    return result.get(
        "text",
        ""
    )


In [ ]:
for idx in tqdm(range(len(df))):

    row = df.iloc[idx]

    audio_path = row["audio_path"]

    if audio_path in processed:
        continue

    try:

        info = sf.info(audio_path)

        duration = info.duration

        t0 = time.time()

        pred_text = transcribe_vosk(
            audio_path,
            model
        )

        t1 = time.time()

        inference_time = t1 - t0

        new_row = {

            "audio_path": audio_path,

            "reference_raw": row["sentence"],

            "prediction_raw": pred_text,

            "audio_duration": duration,

            "inference_time": inference_time
        }

        results_df = pd.concat(
            [
                results_df,
                pd.DataFrame([new_row])
            ],
            ignore_index=True
        )

        processed.add(audio_path)

        # checkpoint every 50 samples
        if len(results_df) % 50 == 0:

            results_df.to_csv(
                CHECKPOINT_PATH,
                index=False
            )

            print(
                f"Checkpoint saved "
                f"({len(results_df)} samples)"
            )

    except Exception as e:

        print(
            f"\nFAILED: {audio_path}"
        )

        print(e)

  0%|          | 0/10519 [00:00<?, ?it/s]/tmp/ipykernel_905/2207672906.py:40: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat(
  0%|          | 50/10519 [02:36<8:47:44,  3.02s/it]

Checkpoint saved (50 samples)


  1%|          | 100/10519 [04:27<6:54:12,  2.39s/it]

Checkpoint saved (100 samples)


  1%|▏         | 150/10519 [06:19<5:22:17,  1.86s/it]

...

 99%|█████████▉| 10450/10519 [5:28:27<02:03,  1.79s/it]

Checkpoint saved (10450 samples)


100%|█████████▉| 10500/10519 [5:29:50<00:26,  1.37s/it]

Checkpoint saved (10500 samples)


100%|██████████| 10519/10519 [5:30:40<00:00,  1.89s/it]


In [ ]:
results_df.to_csv(
    CHECKPOINT_PATH,
    index=False
)

print("Final checkpoint saved.")
print(CHECKPOINT_PATH)

Final checkpoint saved.
/content/drive/MyDrive/asr_project/vosk_checkpoint.csv


In [ ]:
pip -q install hazm jiwer

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 85.7 MB/s eta 0:00:00


In [ ]:
import re
import pandas as pd
import hazm

from jiwer import wer, cer

In [ ]:
CHECKPOINT_PATH = (
    "/content/drive/MyDrive/asr_project/"
    "vosk_checkpoint.csv"
)

df = pd.read_csv(CHECKPOINT_PATH)

print("Loaded:", len(df))


Loaded: 10519


In [ ]:
normalizer = hazm.Normalizer()

PERSIAN_CHAR_MAP = {
    "ك": "ک",
    "ي": "ی",
    "ى": "ی",
    "ؤ": "و",
    "ئ": "ی",
    "ة": "ه",
}

def normalize_persian(text):

    text = str(text)

    text = normalizer.normalize(text)

    for k, v in PERSIAN_CHAR_MAP.items():
        text = text.replace(k, v)

    # number normalization
    text = re.sub(
        r"\d+",
        " <NUM> ",
        text
    )

    text = re.sub(
        r"[^\w\s<>]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text

In [ ]:
df["ref_norm"] = (
    df["reference_raw"]
    .apply(normalize_persian)
)

df["hyp_norm"] = (
    df["prediction_raw"]
    .apply(normalize_persian)
)

In [ ]:
refs = df["ref_norm"].tolist()
hyps = df["hyp_norm"].tolist()

wer_score = wer(
    refs,
    hyps
)

cer_score = cer(
    refs,
    hyps
)

total_audio_duration = (
    df["audio_duration"].sum()
)

total_inference_time = (
    df["inference_time"].sum()
)

rtf = (
    total_inference_time
    /
    total_audio_duration
)

avg_latency = (
    df["inference_time"].mean()
)

# CPU model
peak_memory = 0

In [ ]:
print("\n=========== FINAL RESULTS ===========\n")

print("Model: Vosk Persian")

print(f"Samples: {len(df)}")

print(f"WER: {wer_score:.4f}")

print(f"CER: {cer_score:.4f}")

print(f"RTF: {rtf:.4f}")

print(
    f"Avg latency (s): "
    f"{avg_latency:.4f}"
)

print(
    f"Total inference time (s): "
    f"{total_inference_time:.2f}"
)

print(
    f"Total audio duration (s): "
    f"{total_audio_duration:.2f}"
)

print(
    f"Peak GPU memory (GB): "
    f"{peak_memory:.2f}"
)

print("\n====================================\n")



=========== FINAL RESULTS ===========

Model: Vosk Persian
Samples: 10519
WER: 0.3543
CER: 0.2566
RTF: 0.3051
Avg latency (s): 1.5098
Total inference time (s): 15881.76
Total audio duration (s): 52047.28
Peak GPU memory (GB): 0.00




In [ ]:
summary_df = pd.DataFrame([{

    "model": "Vosk Persian",

    "samples": len(df),

    "wer": wer_score,

    "cer": cer_score,

    "rtf": rtf,

    "avg_latency_sec": avg_latency,

    "gpu_memory_gb": peak_memory,

    "total_inference_time_sec": total_inference_time

}])

summary_df.to_csv(
    "/content/drive/MyDrive/asr_project/vosk_summary.csv",
    index=False
)

print("Summary saved.")

Summary saved.


In [ ]:
import pandas as pd
import re
import unicodedata
from jiwer import wer, cer

CHECKPOINT_PATH = "/content/drive/MyDrive/asr_project/vosk_checkpoint.csv"

df = pd.read_csv(CHECKPOINT_PATH)

df = df.dropna(subset=["reference_raw", "prediction_raw"]).reset_index(drop=True)

In [ ]:
df.head()

,audio_path,reference_raw,prediction_raw,audio_duration,inference_time
0,/content/drive/MyDrive/asr_project/asr_data/fa...,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,2.753750,17.422447
1,/content/drive/MyDrive/asr_project/asr_data/fa...,دعا خوان,دعاخوان,3.103500,0.914036
2,/content/drive/MyDrive/asr_project/asr_data/fa...,اعتماد کرد,اعتماد کرد,4.651043,1.470268
3,/content/drive/MyDrive/asr_project/asr_data/fa...,خب ، تو چیكار می كنی؟,خب تو چیکار می‌کنی,3.701625,1.054197
4,/content/drive/MyDrive/asr_project/asr_data/fa...,آه، نه اصلاُ!,اوه نه اصلا,2.429625,0.902153


In [ ]:
df_sub = df[['prediction_raw', 'reference_raw']]
df_sub.head()

,prediction_raw,reference_raw
0,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم
1,دعاخوان,دعا خوان
2,اعتماد کرد,اعتماد کرد
3,خب تو چیکار می‌کنی,خب ، تو چیكار می كنی؟
4,اوه نه اصلا,آه، نه اصلاُ!


In [ ]:
SKIP = set([
    "ā", "š", "="
])

REPLACEMENTS = {
    "أ": "ا",
    "ك": "ک",
    "ي": "ی",
    "ى": "ی",
    "ﯽ": "ی",
    "ﻮ": "و",
    "ە": "ه",
    "ۀ": "ه",
}

DISCARD = set([
    "!", '"', "#", "&", "'", "(", ")", ",", "-", ".", ":",
    ";", "؟", "،", "؛", "ـ", "…", "«", "»", "–",
    "ً", "ٌ", "َ", "ُ", "ِ", "ّ", "ْ", "ٔ"
])


def nemo_paper_normalize(text: str) -> str:
    if text is None:
        return ""

    text = str(text)

    # Unicode normalization FIRST (important)
    text = unicodedata.normalize("NFKC", text)

    # remove hashtags (NeMo-style filtering)
    text = " ".join(w for w in text.split() if not w.startswith("#"))

    # character replacements
    for k, v in REPLACEMENTS.items():
        text = text.replace(k, v)

    # remove punctuation tokens
    for tok in DISCARD:
        text = text.replace(tok, " ")

    # remove Arabic letter variations like hamza
    text = text.replace("ء", "")

    # collapse whitespace
    text = " ".join(text.split())

    return text

In [ ]:
df_sub["hyp_norm"] = df_sub["prediction_raw"].apply(nemo_paper_normalize)
df_sub["ref_norm"] = df_sub["reference_raw"].apply(nemo_paper_normalize)

/tmp/ipykernel_569/1633250410.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sub["hyp_norm"] = df_sub["prediction_raw"].apply(nemo_paper_normalize)
/tmp/ipykernel_569/1633250410.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sub["ref_norm"] = df_sub["reference_raw"].apply(nemo_paper_normalize)


In [ ]:
df_sub.loc[df_sub["hyp_norm"] == "", "hyp_norm"] = " "
df_sub.loc[df_sub["ref_norm"] == "", "ref_norm"] = " "

In [ ]:
wer_score = wer(df_sub["ref_norm"].tolist(), df_sub["hyp_norm"].tolist())
cer_score = cer(df_sub["ref_norm"].tolist(), df_sub["hyp_norm"].tolist())

print("NeMo-replicated WER:", wer_score)
print("NeMo-replicated CER:", cer_score)
print("Samples:", len(df_sub))

NeMo-replicated WER: 0.2538651370639632
NeMo-replicated CER: 0.1858016278485446
Samples: 10492


In [ ]:
pip install kaldialign

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 4.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import kaldialign


def _alignment_error_rate(ref_tokens, hyp_tokens):
    """
    Compute error rate using Kaldi/Icefall alignment.

    Returns:
        error_rate, substitutions, deletions, insertions, reference_length
    """
    ali = kaldialign.align(ref_tokens, hyp_tokens, "*")

    subs = dels = ins = 0

    for r, h in ali:
        if r == "*":
            ins += 1
        elif h == "*":
            dels += 1
        elif r != h:
            subs += 1

    n_ref = len(ref_tokens)
    err = (subs + dels + ins) / max(1, n_ref)

    return err, subs, dels, ins, n_ref


def add_wer_cer_columns(
    df,
    ref_col="ref_norm",
    hyp_col="hyp_norm",
):
    """
    Adds per-sample WER and CER columns to a dataframe.

    Returns
    -------
    DataFrame
        Copy of dataframe with:
            wer
            cer
            wer_sub
            wer_del
            wer_ins
            cer_sub
            cer_del
            cer_ins
    """

    df = df.copy()

    wers = []
    cers = []

    wer_subs = []
    wer_dels = []
    wer_inss = []

    cer_subs = []
    cer_dels = []
    cer_inss = []

    for ref, hyp in zip(df[ref_col], df[hyp_col]):

        ref = "" if pd.isna(ref) else str(ref)
        hyp = "" if pd.isna(hyp) else str(hyp)

        # ---------- WER ----------
        wer, s, d, i, _ = _alignment_error_rate(
            ref.split(),
            hyp.split()
        )

        wers.append(wer)
        wer_subs.append(s)
        wer_dels.append(d)
        wer_inss.append(i)

        # ---------- CER ----------
        cer, s, d, i, _ = _alignment_error_rate(
            list(ref.replace(" ", "")),
            list(hyp.replace(" ", ""))
        )

        cers.append(cer)
        cer_subs.append(s)
        cer_dels.append(d)
        cer_inss.append(i)

    df["wer"] = wers
    df["cer"] = cers

    df["wer_sub"] = wer_subs
    df["wer_del"] = wer_dels
    df["wer_ins"] = wer_inss

    df["cer_sub"] = cer_subs
    df["cer_del"] = cer_dels
    df["cer_ins"] = cer_inss

    return df


def compute_dataset_wer_cer(
    df,
    ref_col="ref_norm",
    hyp_col="hyp_norm",
):
    """
    Computes overall dataset WER/CER (Icefall/Kaldi style).

    Returns
    -------
    dict
    """

    total_word_sub = total_word_del = total_word_ins = 0
    total_words = 0

    total_char_sub = total_char_del = total_char_ins = 0
    total_chars = 0

    for ref, hyp in zip(df[ref_col], df[hyp_col]):

        ref = "" if pd.isna(ref) else str(ref)
        hyp = "" if pd.isna(hyp) else str(hyp)

        _, s, d, i, n = _alignment_error_rate(
            ref.split(),
            hyp.split()
        )

        total_word_sub += s
        total_word_del += d
        total_word_ins += i
        total_words += n

        _, s, d, i, n = _alignment_error_rate(
            list(ref.replace(" ", "")),
            list(hyp.replace(" ", ""))
        )

        total_char_sub += s
        total_char_del += d
        total_char_ins += i
        total_chars += n

    wer = (
        total_word_sub + total_word_del + total_word_ins
    ) / max(1, total_words)

    cer = (
        total_char_sub + total_char_del + total_char_ins
    ) / max(1, total_chars)

    return {
        "WER": wer,
        "CER": cer,
        "word_sub": total_word_sub,
        "word_del": total_word_del,
        "word_ins": total_word_ins,
        "char_sub": total_char_sub,
        "char_del": total_char_del,
        "char_ins": total_char_ins,
        "total_words": total_words,
        "total_chars": total_chars,
    }

In [ ]:
# Add per-utterance metrics
df_sub = add_wer_cer_columns(df_sub)

# Inspect individual utterances
df_sub[["hyp_norm", "ref_norm", "wer", "cer"]].head()

,prediction_raw,reference_raw,hyp_norm,ref_norm,wer,cer,wer_sub,wer_del,wer_ins,cer_sub,cer_del,cer_ins
0,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,0.000000,0.000000,0,0,0,0,0,0
1,دعاخوان,دعا خوان,دعاخوان,دعا خوان,1.000000,0.000000,1,1,0,0,0,0
2,اعتماد کرد,اعتماد کرد,اعتماد کرد,اعتماد کرد,0.000000,0.000000,0,0,0,0,0,0
3,خب تو چیکار می‌کنی,خب ، تو چیكار می كنی؟,خب تو چیکار می‌کنی,خب تو چیکار می کنی,0.400000,0.071429,1,1,0,0,0,1
4,اوه نه اصلا,آه، نه اصلاُ!,اوه نه اصلا,آه نه اصلا,0.333333,0.250000,1,0,0,1,0,1


In [ ]:
def compute_dataset_wer_cer(
    df,
    ref_col="ref_norm",
    hyp_col="hyp_norm",
):
    """
    Computes overall dataset WER/CER (Icefall/Kaldi style).

    Returns
    -------
    dict
    """

    total_word_sub = total_word_del = total_word_ins = 0
    total_words = 0

    total_char_sub = total_char_del = total_char_ins = 0
    total_chars = 0

    for ref, hyp in zip(df[ref_col], df[hyp_col]):

        ref = "" if pd.isna(ref) else str(ref)
        hyp = "" if pd.isna(hyp) else str(hyp)

        _, s, d, i, n = _alignment_error_rate(
            ref.split(),
            hyp.split()
        )

        total_word_sub += s
        total_word_del += d
        total_word_ins += i
        total_words += n

        _, s, d, i, n = _alignment_error_rate(
            list(ref.replace(" ", "")),
            list(hyp.replace(" ", ""))
        )

        total_char_sub += s
        total_char_del += d
        total_char_ins += i
        total_chars += n

    wer = (
        total_word_sub + total_word_del + total_word_ins
    ) / max(1, total_words)

    cer = (
        total_char_sub + total_char_del + total_char_ins
    ) / max(1, total_chars)

    return {
        "WER": wer,
        "CER": cer,
        "word_sub": total_word_sub,
        "word_del": total_word_del,
        "word_ins": total_word_ins,
        "char_sub": total_char_sub,
        "char_del": total_char_del,
        "char_ins": total_char_ins,
        "total_words": total_words,
        "total_chars": total_chars,
    }

compute_dataset_wer_cer(df_sub)

{'WER': 0.22337633482843375,
 'CER': 0.0592807158426064,
 'word_sub': 11186,
 'word_del': 3783,
 'word_ins': 1117,
 'char_sub': 4040,
 'char_del': 4512,
 'char_ins': 7984,
 'total_words': 72013,
 'total_chars': 278944}

### results

In [ ]:
results = compute_dataset_wer_cer(df_sub)

print(f"WER: {results['WER']:.4f}")
print(f"CER: {results['CER']:.4f}")
print(results)

WER: 0.2234
CER: 0.0593
{'WER': 0.22337633482843375, 'CER': 0.0592807158426064, 'word_sub': 11186, 'word_del': 3783, 'word_ins': 1117, 'char_sub': 4040, 'char_del': 4512, 'char_ins': 7984, 'total_words': 72013, 'total_chars': 278944}


In [ ]:
df_sub[df_sub["ref_norm"].str.match(r"^(?=.*\d)(?=.*[^\d]).+", na=False)]

,prediction_raw,reference_raw,hyp_norm,ref_norm,wer,cer,wer_sub,wer_del,wer_ins,cer_sub,cer_del,cer_ins
1688,هی این خانوم ایده جالبی داره بگذار گوش کنیم,هی، این خانم ایده جالبی دارد بگذار گوش کنیم\t\...,هی این خانوم ایده جالبی داره بگذار گوش کنیم,هی این خانم ایده جالبی دارد بگذار گوش کنیم 2 0...,0.997629,0.999308,2,2943,0,0,50565,0


In [ ]:
df_sub_1 = df_sub[df_sub['cer'] > df_sub['wer']]
df_sub_1

,prediction_raw,reference_raw,hyp_norm,ref_norm,wer,cer,wer_sub,wer_del,wer_ins,cer_sub,cer_del,cer_ins
21,نه گوشی سامسونگ آم هنوزم تلویزیون,هیچ گل مینا دارید؟,نه گوشی سامسونگ آم هنوزم تلویزیون,هیچ گل مینا دارید,1.500000,1.571429,4,0,2,8,0,14
102,نه شروع به لخته شدن می‌کنم,بلافاصله شروع به لخته شدن میکند,نه شروع به لخته شدن می‌کنم,بلافاصله شروع به لخته شدن میکند,0.333333,0.346154,2,0,0,2,6,1
106,هان فقط رفتن تو بزرگراه و گشت زدن,(خنده) فقط، رفتن تو بزرگراه و گشت زدن,هان فقط رفتن تو بزرگراه و گشت زدن,خنده فقط رفتن تو بزرگراه و گشت زدن,0.125000,0.148148,1,0,0,1,2,1
298,دستمال گشود و به سقف خیره شد,چشمانش را گشود و به سقف خیره شد.,دستمال گشود و به سقف خیره شد,چشمانش را گشود و به سقف خیره شد,0.250000,0.291667,1,1,0,3,3,1
355,جشن پنجاه پنجاه‌ساله,جشن پنجاه ساله,جشن پنجاه پنجاه‌ساله,جشن پنجاه ساله,0.333333,0.500000,1,0,0,0,0,6
397,ال دو نقطه روز زن چی,ال:روز زن چی؟,ال دو نقطه روز زن چی,ال روز زن چی,0.500000,0.666667,0,0,2,0,0,6
509,هدفش این بود که در آمریکا,هدفش این بود که در جامعه آمریکا,هدفش این بود که در آمریکا,هدفش این بود که در جامعه آمریکا,0.142857,0.200000,0,1,0,0,5,0
532,روز مکان و حتی زمان مراجعه شما را تعیین می‌کند,روز ، مکان و حتی زمان مراجعه شما را تعیین کنند.,روز مکان و حتی زمان مراجعه شما را تعیین می‌کند,روز مکان و حتی زمان مراجعه شما را تعیین کنند,0.100000,0.114286,1,0,0,0,1,3
555,خب چرا نباید باشم فکر می‌کنم,خب ، چرا نباید باشم ؟,خب چرا نباید باشم فکر می‌کنم,خب چرا نباید باشم,0.500000,0.642857,0,0,2,0,0,9
920,این من برمیگرده به چند سال پیش,این داستان من برمیگرده به چند سال پیش,این من برمیگرده به چند سال پیش,این داستان من برمیگرده به چند سال پیش,0.125000,0.200000,0,1,0,0,6,0


In [ ]:
df_sub = df_sub.drop(index=1688)
df_sub.count()

,0
prediction_raw,10491
reference_raw,10491
hyp_norm,10491
ref_norm,10491
wer,10491
cer,10491
wer_sub,10491
wer_del,10491
wer_ins,10491
cer_sub,10491


# Nemo

In [ ]:
!pip install -q pandas==2.2.2
!pip install -q numpy==1.26.4
!pip install -q numba==0.60.0
!pip install -q nemo_toolkit['asr']

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 76.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=

In [ ]:
!pip install -q jiwer
!pip install -q hazm
!pip install -q soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 52.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.6 MB/s eta 0:00:00


In [ ]:
import os
import re
import time
import random
import numpy as np
import pandas as pd
import torch
import soundfile as sf
from tqdm import tqdm
from jiwer import wer, cer
import nemo.collections.asr as nemo_asr
import hazm

[NeMo W 2026-06-25 10:19:41 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
[NeMo W 2026-06-25 10:19:43 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
      m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
    
[NeMo W 2026-06-25 10:19:43 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
      m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
    
[NeMo W 2026-06-25 10:19:43 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(flt)p?( \(default\))?$', token):
    
[NeMo W 2026-06-25 10:19:43 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(dbl)p?( \(default\))?$', token):
    


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
data_root = "/content/drive/MyDrive/asr_project/asr_data/fa_test_only"
tsv_path = os.path.join(data_root, "test.tsv")
audio_root = os.path.join(data_root, "clips")

df = pd.read_csv(tsv_path, sep="\t")
df = df.dropna(subset=["path", "sentence"]).reset_index(drop=True)

df["audio_path"] = df["path"].apply(lambda x: os.path.join(audio_root, x))

print("Total samples:", len(df))

Total samples: 10519


In [ ]:
normalizer = hazm.Normalizer()

PERSIAN_CHAR_MAP = {
    "ك": "ک",
    "ي": "ی",
    "ى": "ی",
    "ؤ": "و",
    "ئ": "ی",
    "ة": "ه",
}

def normalize_persian(text):
    text = str(text)
    text = normalizer.normalize(text)

    for k, v in PERSIAN_CHAR_MAP.items():
        text = text.replace(k, v)

    text = re.sub(r"\d+", " <NUM> ", text)   # number handling

    text = re.sub(r"[^\w\s<>]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
MODEL_NAME = "nvidia/stt_fa_fastconformer_hybrid_large"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = nemo_asr.models.ASRModel.from_pretrained(MODEL_NAME)
model = model.to(device)

print("Device:", device)

stt_fa_fastconformer_hybrid_large.nemo:   0%|          | 0.00/459M [00:00<?, ?B/s]

[NeMo I 2026-06-19 07:01:23 mixins:184] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-06-19 07:01:24 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: dummy
    sample_rate: 16000
    batch_size: 1
    shuffle: true
    num_workers: 8
    pin_memory: true
    max_duration: 10
    min_duration: 0.5
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: fully_randomized
    bucketing_batch_size: null
    use_lhotse: true
    lhotse:
      shar_path: /data_artifacts/data/shar/train
      batch_duration: 1200
      quadratic_duration: 15
      num_buckets: 10
      num_cuts_for_bins_estimate: 10000
      buffer_size: 10000
      shuffle_buffer_size: 10000
    
[NeMo W 2026-06-19 07:01:24 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a vali

[NeMo I 2026-06-19 07:01:25 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-06-19 07:01:25 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-06-19 07:01:25 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-06-19 07:01:26 save_restore_connector:285] Model EncDecHybridRNNTCTCBPEModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--stt_fa_fastconformer_hybrid_large/snapshots/249cf5bf70dda7220a60ddeeecff2f6aad8e1784/stt_fa_fastconformer_hybrid_large.nemo.
Device: cuda


In [ ]:
CHECKPOINT_PATH = "/content/drive/MyDrive/asr_project/nemo_checkpoint.csv"

if os.path.exists(CHECKPOINT_PATH):
    results_df = pd.read_csv(CHECKPOINT_PATH)
    processed = set(results_df["audio_path"].tolist())
    print("Resuming from:", len(results_df), "samples")
else:
    results_df = pd.DataFrame(columns=[
        "audio_path",
        "reference_raw",
        "prediction_raw",
        "audio_duration",
        "inference_time"
    ])
    processed = set()

In [ ]:
torch.cuda.empty_cache()
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()

total_audio_duration = 0.0
total_inference_time = 0.0

start_time = time.time()

for idx in tqdm(range(len(df))):

    row = df.iloc[idx]
    audio_path = row["audio_path"]

    if audio_path in processed:
        continue

    try:
        # -------------------------
        # duration
        # -------------------------
        info = sf.info(audio_path)
        duration = info.frames / info.samplerate
        total_audio_duration += duration

        # -------------------------
        # inference timing
        # -------------------------
        t0 = time.time()

        pred = model.transcribe([audio_path], batch_size=1)[0]

        t1 = time.time()
        inf_time = t1 - t0
        total_inference_time += inf_time

        # -------------------------
        # store RAW ONLY
        # -------------------------
        new_row = {
            "audio_path": audio_path,
            "reference_raw": row["sentence"],
            "prediction_raw": pred,
            "audio_duration": duration,
            "inference_time": inf_time
        }

        results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)
        processed.add(audio_path)

        # -------------------------
        # checkpoint
        # -------------------------
        if len(results_df) % 50 == 0:
            results_df.to_csv(CHECKPOINT_PATH, index=False)
            print(f"Checkpoint saved: {len(results_df)} samples")

    except Exception as e:
        print("FAILED:", audio_path)
        print(e)

  0%|          | 0/10519 [00:00<?, ?it/s][NeMo W 2026-06-19 07:09:40 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-19 07:09:40 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)

Transcribing: 1it [00:00, 17.35it/s]
  2%|▏         | 169/10519 [00:00<00:08, 1286.35it/s][NeMo W 2026-06-19 07:09:41 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-19 07:09:41 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tok

Checkpoint saved: 200 samples


[NeMo W 2026-06-19 07:10:12 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-19 07:10:12 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)

Transcribing: 1it [00:00, 16.82it/s]
[NeMo W 2026-06-19 07:10:13 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-19 07:10:13 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your to

Checkpoint saved: 250 samples


[NeMo W 2026-06-19 07:11:00 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-19 07:11:00 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)

Transcribing: 1it [00:00, 17.54it/s]
[NeMo W 2026-06-19 07:11:01 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-19 07:11:01 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your to

...

[NeMo W 2026-06-19 09:56:54 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-19 09:56:54 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)

Transcribing: 1it [00:00, 12.49it/s]
 99%|█████████▉| 10401/10519 [2:47:13<02:43,  1.38s/it][NeMo W 2026-06-19 09:56:55 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-19 09:56:55 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) p

Checkpoint saved: 10450 samples


[NeMo W 2026-06-19 09:57:44 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-19 09:57:44 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)

Transcribing: 1it [00:00, 11.74it/s]
 99%|█████████▉| 10451/10519 [2:48:03<01:31,  1.34s/it][NeMo W 2026-06-19 09:57:44 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-19 09:57:44 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) p

Checkpoint saved: 10500 samples


[NeMo W 2026-06-19 09:58:33 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-19 09:58:33 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)

Transcribing: 0it [00:00, ?it/s]
Transcribing: 1it [00:00,  9.16it/s]
100%|█████████▉| 10501/10519 [2:48:52<00:21,  1.21s/it][NeMo W 2026-06-19 09:58:34 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-19 09:58:34 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenizat

In [ ]:
results_df.to_csv(CHECKPOINT_PATH, index=False)
print("Final checkpoint saved.")

In [ ]:
# normalize AFTER full run
results_df["ref_norm"] = results_df["reference_raw"].apply(normalize_persian)
results_df["hyp_norm"] = results_df["prediction_raw"].apply(normalize_persian)

refs = results_df["ref_norm"].tolist()
hyps = results_df["hyp_norm"].tolist()

wer_score = wer(refs, hyps)
cer_score = cer(refs, hyps)

In [ ]:
if device == "cuda":
    peak_memory = torch.cuda.max_memory_allocated() / 1024**3
else:
    peak_memory = 0

In [ ]:
print("\n=========== FINAL RESULTS ===========\n")

print(f"Samples: {len(results_df)}")
print(f"WER: {wer_score:.4f}")
print(f"CER: {cer_score:.4f}")
print(f"RTF: {rtf:.4f}")
print(f"Avg latency (s): {avg_latency:.4f}")
print(f"Total inference time (s): {total_inference_time:.2f}")
print(f"Total audio duration (s): {total_audio_duration:.2f}")
print(f"Peak GPU memory (GB): {peak_memory:.2f}")

print("\n====================================\n")

In [ ]:
import pandas as pd
import re
import unicodedata
from jiwer import wer, cer

CHECKPOINT_PATH = "/content/drive/MyDrive/asr_project/nemo_checkpoint.csv"

df = pd.read_csv(CHECKPOINT_PATH)

df = df.dropna(subset=["reference_raw", "prediction_raw"]).reset_index(drop=True)

In [ ]:
def extract_nemo_text(x):
    if pd.isna(x):
        return None
    x = str(x)

    # strict extraction of "text='...'"
    match = re.search(r"text='(.*?)'", x)
    if match:
        return match.group(1)

    return None


df["hyp"] = df["prediction_raw"].apply(extract_nemo_text)
df["ref"] = df["reference_raw"].astype(str)

In [ ]:
df_sub = df[['hyp', 'ref']]
df_sub.head()

,hyp,ref
0,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم
1,داغان,دعا خوان
2,اعتماد کرد,اعتماد کرد
3,خب تو چیکار می کنی,خب ، تو چیكار می كنی؟
4,آه نه اصلا,آه، نه اصلاُ!


In [ ]:
SKIP = set([
    "ā", "š", "="
])

REPLACEMENTS = {
    "أ": "ا",
    "ك": "ک",
    "ي": "ی",
    "ى": "ی",
    "ﯽ": "ی",
    "ﻮ": "و",
    "ە": "ه",
    "ۀ": "ه",
}

DISCARD = set([
    "!", '"', "#", "&", "'", "(", ")", ",", "-", ".", ":",
    ";", "؟", "،", "؛", "ـ", "…", "«", "»", "–",
    "ً", "ٌ", "َ", "ُ", "ِ", "ّ", "ْ", "ٔ"
])


def nemo_paper_normalize(text: str) -> str:
    if text is None:
        return ""

    text = str(text)

    # Unicode normalization FIRST (important)
    text = unicodedata.normalize("NFKC", text)

    # remove hashtags (NeMo-style filtering)
    text = " ".join(w for w in text.split() if not w.startswith("#"))

    # character replacements
    for k, v in REPLACEMENTS.items():
        text = text.replace(k, v)

    # remove punctuation tokens
    for tok in DISCARD:
        text = text.replace(tok, " ")

    # remove Arabic letter variations like hamza
    text = text.replace("ء", "")

    # collapse whitespace
    text = " ".join(text.split())

    return text

In [ ]:
df_sub["hyp_norm"] = df_sub["hyp"].apply(nemo_paper_normalize)
df_sub["ref_norm"] = df_sub["ref"].apply(nemo_paper_normalize)

/tmp/ipykernel_931/3243310558.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sub["hyp_norm"] = df_sub["hyp"].apply(nemo_paper_normalize)
/tmp/ipykernel_931/3243310558.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sub["ref_norm"] = df_sub["ref"].apply(nemo_paper_normalize)


In [ ]:
df_sub.head()

,hyp,ref,hyp_norm,ref_norm
0,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم
1,داغان,دعا خوان,داغان,دعا خوان
2,اعتماد کرد,اعتماد کرد,اعتماد کرد,اعتماد کرد
3,خب تو چیکار می کنی,خب ، تو چیكار می كنی؟,خب تو چیکار می کنی,خب تو چیکار می کنی
4,آه نه اصلا,آه، نه اصلاُ!,آه نه اصلا,آه نه اصلا


In [ ]:
df_sub.loc[df_sub["hyp_norm"] == "", "hyp_norm"] = " "
df_sub.loc[df_sub["ref_norm"] == "", "ref_norm"] = " "

In [ ]:
df_sub.head()

,hyp,ref,hyp_norm,ref_norm
0,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم
1,داغان,دعا خوان,داغان,دعا خوان
2,اعتماد کرد,اعتماد کرد,اعتماد کرد,اعتماد کرد
3,خب تو چیکار می کنی,خب ، تو چیكار می كنی؟,خب تو چیکار می کنی,خب تو چیکار می کنی
4,آه نه اصلا,آه، نه اصلاُ!,آه نه اصلا,آه نه اصلا


In [ ]:
wer_score = wer(df_sub["ref_norm"].tolist(), df_sub["hyp_norm"].tolist())
cer_score = cer(df_sub["ref_norm"].tolist(), df_sub["hyp_norm"].tolist())

print("NeMo-replicated WER:", wer_score)
print("NeMo-replicated CER:", cer_score)
print("Samples:", len(df_sub))

NeMo-replicated WER: 0.07659851993824203
NeMo-replicated CER: 0.14771397451809823
Samples: 10519


In [ ]:
df_sub.head()

,hyp,ref,hyp_norm,ref_norm
0,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم
1,داغان,دعا خوان,داغان,دعا خوان
2,اعتماد کرد,اعتماد کرد,اعتماد کرد,اعتماد کرد
3,خب تو چیکار می کنی,خب ، تو چیكار می كنی؟,خب تو چیکار می کنی,خب تو چیکار می کنی
4,آه نه اصلا,آه، نه اصلاُ!,آه نه اصلا,آه نه اصلا


In [ ]:
pip install kaldialign

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 8.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import kaldialign


def _alignment_error_rate(ref_tokens, hyp_tokens):
    """
    Compute error rate using Kaldi/Icefall alignment.

    Returns:
        error_rate, substitutions, deletions, insertions, reference_length
    """
    ali = kaldialign.align(ref_tokens, hyp_tokens, "*")

    subs = dels = ins = 0

    for r, h in ali:
        if r == "*":
            ins += 1
        elif h == "*":
            dels += 1
        elif r != h:
            subs += 1

    n_ref = len(ref_tokens)
    err = (subs + dels + ins) / max(1, n_ref)

    return err, subs, dels, ins, n_ref


def add_wer_cer_columns(
    df,
    ref_col="ref_norm",
    hyp_col="hyp_norm",
):
    """
    Adds per-sample WER and CER columns to a dataframe.

    Returns
    -------
    DataFrame
        Copy of dataframe with:
            wer
            cer
            wer_sub
            wer_del
            wer_ins
            cer_sub
            cer_del
            cer_ins
    """

    df = df.copy()

    wers = []
    cers = []

    wer_subs = []
    wer_dels = []
    wer_inss = []

    cer_subs = []
    cer_dels = []
    cer_inss = []

    for ref, hyp in zip(df[ref_col], df[hyp_col]):

        ref = "" if pd.isna(ref) else str(ref)
        hyp = "" if pd.isna(hyp) else str(hyp)

        # ---------- WER ----------
        wer, s, d, i, _ = _alignment_error_rate(
            ref.split(),
            hyp.split()
        )

        wers.append(wer)
        wer_subs.append(s)
        wer_dels.append(d)
        wer_inss.append(i)

        # ---------- CER ----------
        cer, s, d, i, _ = _alignment_error_rate(
            list(ref.replace(" ", "")),
            list(hyp.replace(" ", ""))
        )

        cers.append(cer)
        cer_subs.append(s)
        cer_dels.append(d)
        cer_inss.append(i)

    df["wer"] = wers
    df["cer"] = cers

    df["wer_sub"] = wer_subs
    df["wer_del"] = wer_dels
    df["wer_ins"] = wer_inss

    df["cer_sub"] = cer_subs
    df["cer_del"] = cer_dels
    df["cer_ins"] = cer_inss

    return df


def compute_dataset_wer_cer(
    df,
    ref_col="ref_norm",
    hyp_col="hyp_norm",
):
    """
    Computes overall dataset WER/CER (Icefall/Kaldi style).

    Returns
    -------
    dict
    """

    total_word_sub = total_word_del = total_word_ins = 0
    total_words = 0

    total_char_sub = total_char_del = total_char_ins = 0
    total_chars = 0

    for ref, hyp in zip(df[ref_col], df[hyp_col]):

        ref = "" if pd.isna(ref) else str(ref)
        hyp = "" if pd.isna(hyp) else str(hyp)

        _, s, d, i, n = _alignment_error_rate(
            ref.split(),
            hyp.split()
        )

        total_word_sub += s
        total_word_del += d
        total_word_ins += i
        total_words += n

        _, s, d, i, n = _alignment_error_rate(
            list(ref.replace(" ", "")),
            list(hyp.replace(" ", ""))
        )

        total_char_sub += s
        total_char_del += d
        total_char_ins += i
        total_chars += n

    wer = (
        total_word_sub + total_word_del + total_word_ins
    ) / max(1, total_words)

    cer = (
        total_char_sub + total_char_del + total_char_ins
    ) / max(1, total_chars)

    return {
        "WER": wer,
        "CER": cer,
        "word_sub": total_word_sub,
        "word_del": total_word_del,
        "word_ins": total_word_ins,
        "char_sub": total_char_sub,
        "char_del": total_char_del,
        "char_ins": total_char_ins,
        "total_words": total_words,
        "total_chars": total_chars,
    }

In [ ]:
# Add per-utterance metrics
df_sub = add_wer_cer_columns(df_sub)

# Inspect individual utterances
df_sub[["hyp_norm", "ref_norm", "wer", "cer"]].head()

,hyp_norm,ref_norm,wer,cer
0,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,0.0,0.000000
1,داغان,دعا خوان,1.0,0.428571
2,اعتماد کرد,اعتماد کرد,0.0,0.000000
3,خب تو چیکار می کنی,خب تو چیکار می کنی,0.0,0.000000
4,آه نه اصلا,آه نه اصلا,0.0,0.000000


In [ ]:
def compute_dataset_wer_cer(
    df,
    ref_col="ref",
    hyp_col="hyp",
):
    """
    Computes overall dataset WER/CER (Icefall/Kaldi style).

    Returns
    -------
    dict
    """

    total_word_sub = total_word_del = total_word_ins = 0
    total_words = 0

    total_char_sub = total_char_del = total_char_ins = 0
    total_chars = 0

    for ref, hyp in zip(df[ref_col], df[hyp_col]):

        ref = "" if pd.isna(ref) else str(ref)
        hyp = "" if pd.isna(hyp) else str(hyp)

        _, s, d, i, n = _alignment_error_rate(
            ref.split(),
            hyp.split()
        )

        total_word_sub += s
        total_word_del += d
        total_word_ins += i
        total_words += n

        _, s, d, i, n = _alignment_error_rate(
            list(ref.replace(" ", "")),
            list(hyp.replace(" ", ""))
        )

        total_char_sub += s
        total_char_del += d
        total_char_ins += i
        total_chars += n

    wer = (
        total_word_sub + total_word_del + total_word_ins
    ) / max(1, total_words)

    cer = (
        total_char_sub + total_char_del + total_char_ins
    ) / max(1, total_chars)

    return {
        "WER": wer,
        "CER": cer,
        "word_sub": total_word_sub,
        "word_del": total_word_del,
        "word_ins": total_word_ins,
        "char_sub": total_char_sub,
        "char_del": total_char_del,
        "char_ins": total_char_ins,
        "total_words": total_words,
        "total_chars": total_chars,
    }

compute_dataset_wer_cer(df_sub)

{'WER': 0.12325255967967635,
 'CER': 0.03793886462882096,
 'word_sub': 7659,
 'word_del': 1006,
 'word_ins': 231,
 'char_sub': 1382,
 'char_del': 9152,
 'char_ins': 326,
 'total_words': 72177,
 'total_chars': 286250}

### results

In [ ]:
results = compute_dataset_wer_cer(df_sub)

print(f"WER: {results['WER']:.4f}")
print(f"CER: {results['CER']:.4f}")
print(results)

WER: 0.1233
CER: 0.0379
{'WER': 0.12325255967967635, 'CER': 0.03793886462882096, 'word_sub': 7659, 'word_del': 1006, 'word_ins': 231, 'char_sub': 1382, 'char_del': 9152, 'char_ins': 326, 'total_words': 72177, 'total_chars': 286250}


In [ ]:
df_sub

,hyp,ref,hyp_norm,ref_norm,wer,cer,wer_sub,wer_del,wer_ins,cer_sub,cer_del,cer_ins
0,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,0.000000,0.000000,0,0,0,0,0,0
1,داغان,دعا خوان,داغان,دعا خوان,1.000000,0.428571,1,1,0,1,2,0
2,اعتماد کرد,اعتماد کرد,اعتماد کرد,اعتماد کرد,0.000000,0.000000,0,0,0,0,0,0
3,خب تو چیکار می کنی,خب ، تو چیكار می كنی؟,خب تو چیکار می کنی,خب تو چیکار می کنی,0.000000,0.000000,0,0,0,0,0,0
4,آه نه اصلا,آه، نه اصلاُ!,آه نه اصلا,آه نه اصلا,0.000000,0.000000,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
10514,بخورن گوشت قرمز پروتئین توی این گرما اور می پوشه,بخورن، گوشت قرمز، پروتئین توی این گرما اُور می...,بخورن گوشت قرمز پروتئین توی این گرما اور می پوشه,بخورن گوشت قرمز پروتئین توی این گرما ا ور می پوشه,0.181818,0.000000,1,1,0,0,0,0
10515,خود را از وان بیرون می کشد و کنار وان مینشیند,خود را از وان بیرون می کشد و کنار وان مینشیند,خود را از وان بیرون می کشد و کنار وان مینشیند,خود را از وان بیرون می کشد و کنار وان مینشیند,0.000000,0.000000,0,0,0,0,0,0
10516,هیچ وقت زیر قولت نزن من از آدمای بدقول بدم مییاد,هیچ وقت زیر قولت نزن من از آدمای بدقول بدم مییاد,هیچ وقت زیر قولت نزن من از آدمای بدقول بدم مییاد,هیچ وقت زیر قولت نزن من از آدمای بدقول بدم مییاد,0.000000,0.000000,0,0,0,0,0,0
10517,پدر از تخت پایین میآید عصا را باز میکند,پدر از تخت پایین میآید عصا را باز میکند,پدر از تخت پایین میآید عصا را باز میکند,پدر از تخت پایین میآید عصا را باز میکند,0.000000,0.000000,0,0,0,0,0,0


In [ ]:
df_sub[df_sub["ref_norm"].str.match(r"^(?=.*\d)(?=.*[^\d]).+", na=False)]

,hyp,ref,hyp_norm,ref_norm,wer,cer,wer_sub,wer_del,wer_ins,cer_sub,cer_del,cer_ins


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df_sub)

https://docs.google.com/spreadsheets/d/1Q8zqVEME9yd3XcX58O9CDV2WcWhvVLzvqaU79xxA7-s/edit#gid=0


In [ ]:
df_sub

,hyp,ref,hyp_norm,ref_norm,wer,cer,wer_sub,wer_del,wer_ins,cer_sub,cer_del,cer_ins
0,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,0.000000,0.000000,0,0,0,0,0,0
1,داغان,دعا خوان,داغان,دعا خوان,1.000000,0.428571,1,1,0,1,2,0
2,اعتماد کرد,اعتماد کرد,اعتماد کرد,اعتماد کرد,0.000000,0.000000,0,0,0,0,0,0
3,خب تو چیکار می کنی,خب ، تو چیكار می كنی؟,خب تو چیکار می کنی,خب تو چیکار می کنی,0.000000,0.000000,0,0,0,0,0,0
4,آه نه اصلا,آه، نه اصلاُ!,آه نه اصلا,آه نه اصلا,0.000000,0.000000,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
10514,بخورن گوشت قرمز پروتئین توی این گرما اور می پوشه,بخورن، گوشت قرمز، پروتئین توی این گرما اُور می...,بخورن گوشت قرمز پروتئین توی این گرما اور می پوشه,بخورن گوشت قرمز پروتئین توی این گرما ا ور می پوشه,0.181818,0.000000,1,1,0,0,0,0
10515,خود را از وان بیرون می کشد و کنار وان مینشیند,خود را از وان بیرون می کشد و کنار وان مینشیند,خود را از وان بیرون می کشد و کنار وان مینشیند,خود را از وان بیرون می کشد و کنار وان مینشیند,0.000000,0.000000,0,0,0,0,0,0
10516,هیچ وقت زیر قولت نزن من از آدمای بدقول بدم مییاد,هیچ وقت زیر قولت نزن من از آدمای بدقول بدم مییاد,هیچ وقت زیر قولت نزن من از آدمای بدقول بدم مییاد,هیچ وقت زیر قولت نزن من از آدمای بدقول بدم مییاد,0.000000,0.000000,0,0,0,0,0,0
10517,پدر از تخت پایین میآید عصا را باز میکند,پدر از تخت پایین میآید عصا را باز میکند,پدر از تخت پایین میآید عصا را باز میکند,پدر از تخت پایین میآید عصا را باز میکند,0.000000,0.000000,0,0,0,0,0,0


In [ ]:
df_sub_1 = df_sub[df_sub['cer'] > df_sub['wer']]
df_sub_1

,hyp,ref,hyp_norm,ref_norm,wer,cer,wer_sub,wer_del,wer_ins,cer_sub,cer_del,cer_ins
108,فقط رفتن تو بزرگراه و گشت زدن,(خنده) فقط، رفتن تو بزرگراه و گشت زدن,فقط رفتن تو بزرگراه و گشت زدن,خنده فقط رفتن تو بزرگراه و گشت زدن,0.125000,0.148148,0,1,0,0,4,0
688,در این نزدیکی چشمه آبی هست و من مرتب نوک خود ر...,پاسخ داد : در این نزدیکی چشمه آبی هست و من مرت...,در این نزدیکی چشمه آبی هست و من مرتب نوک خود ر...,پاسخ داد در این نزدیکی چشمه آبی هست و من مرتب ...,0.076923,0.098592,0,2,0,0,7,0
1006,ترامپ گفته بن سلمان بیشتر از من از جنایت نفرت ...,ترامپ گفته بن سلمان بیشتر از من از جنایت نفرت ...,ترامپ گفته بن سلمان بیشتر از من از جنایت نفرت ...,ترامپ گفته بن سلمان بیشتر از من از جنایت نفرت ...,0.090909,0.100000,0,0,1,0,0,4
1495,حالا نمیدونم حکایت این آیه ست که میگه ان مع ال...,حالا نمیدونم حکایت این آیه ست که میگه ان مع ال...,حالا نمیدونم حکایت این آیه ست که میگه ان مع ال...,حالا نمیدونم حکایت این آیه ست که میگه ان مع ال...,0.052632,0.056338,0,1,0,0,4,0
1699,بنابراین در جلسات اغلب کنار دست من مینشست تا ب...,بنابراین در جلسات اغلب کنار دست من مینشست تا ب...,بنابراین در جلسات اغلب کنار دست من مینشست تا ب...,بنابراین در جلسات اغلب کنار دست من مینشست تا ب...,0.285714,0.311475,0,4,0,0,19,0
3096,فکر می کنم بعضی از عکس ها کم نور دیده اند,فکر می کنم بعضی از این عکس ها کم نور دیده اند.,فکر می کنم بعضی از عکس ها کم نور دیده اند,فکر می کنم بعضی از این عکس ها کم نور دیده اند,0.083333,0.088235,0,1,0,0,3,0
3431,احساس مرموزی به من دست داده بود که کسی دارد مر...,احساس مرموزی به من دست داده بود که کسی دارد مر...,احساس مرموزی به من دست داده بود که کسی دارد مر...,احساس مرموزی به من دست داده بود که کسی دارد مر...,0.083333,0.093023,1,0,0,0,4,0
3727,این که همش شد موش فکر کردم اسمت دیو سه سره,با خنده این که همش شد موش. فکر کردم اسمت دیو س...,این که همش شد موش فکر کردم اسمت دیو سه سره,با خنده این که همش شد موش فکر کردم اسمت دیو سه...,0.153846,0.157895,0,2,0,0,6,0
4078,از خودم,از خودم در نمیآورم,از خودم,از خودم در نمیآورم,0.500000,0.600000,0,2,0,0,9,0
4355,تاچشم به آخر اسفندماه شد و باز یه سال بزرگتر ش...,تاچشم به هم بزنی میبینی آخر اسفندماه شد و باز ...,تاچشم به آخر اسفندماه شد و باز یه سال بزرگتر ش...,تاچشم به هم بزنی میبینی آخر اسفندماه شد و باز ...,0.166667,0.171429,0,3,0,0,12,0


In [ ]:
df_sub.count()

,0
hyp,10519
ref,10519
hyp_norm,10519
ref_norm,10519
wer,10519
cer,10519
wer_sub,10519
wer_del,10519
wer_ins,10519
cer_sub,10519


In [ ]:
df_sub = df_sub.drop(index=1701)
df_sub.count()

,0
hyp,10518
ref,10518
hyp_norm,10518
ref_norm,10518
wer,10518
cer,10518
wer_sub,10518
wer_del,10518
wer_ins,10518
cer_sub,10518


In [ ]:
df_sub[df_sub["ref"].str.contains(r"\f", na=False)]

,hyp,ref,hyp_norm,ref_norm,wer,cer,wer_sub,wer_del,wer_ins,cer_sub,cer_del,cer_ins


# w2v

In [ ]:
!pip install -q transformers datasets evaluate jiwer
!pip install -q torchaudio librosa soundfile
!pip install -q hazm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 39.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 72.4 MB/s eta 0:00:00


In [ ]:
import zipfile
import os

ZIP_PATH = "/content/drive/MyDrive/asr_project/fa_test_only.zip"
EXTRACT_PATH = "/content/drive/MyDrive/asr_project/asr_data/"
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
      zip_ref.extractall(EXTRACT_PATH)
      print("Dataset extracted to:", EXTRACT_PATH)

Dataset extracted to: /content/drive/MyDrive/asr_project/asr_data/


In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import torch
import librosa
import soundfile as sf
from tqdm import tqdm

from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/asr_project/asr_data/fa_test_only"

TSV_PATH = os.path.join(DATA_ROOT, "test.tsv")
AUDIO_ROOT = os.path.join(DATA_ROOT, "clips")

df = pd.read_csv(TSV_PATH, sep="\t")
df = df.dropna(subset=["path", "sentence"]).reset_index(drop=True)

df["audio_path"] = df["path"].apply(
    lambda x: os.path.join(AUDIO_ROOT, x)
)

print("Samples:", len(df))

Samples: 10519


In [ ]:
MODEL_NAME = "m3hrdadfi/wav2vec2-large-xlsr-persian-v3"

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_NAME).to(device)

print("Device:", device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/307 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/399 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Device: cuda


In [ ]:
CHECKPOINT_PATH = (
    "/content/drive/MyDrive/asr_project/w2v_persian_checkpoint.csv"
)

if os.path.exists(CHECKPOINT_PATH):

    results_df = pd.read_csv(CHECKPOINT_PATH)
    processed = set(results_df["audio_path"].tolist())
    print("Resuming from:", len(results_df))

else:

    results_df = pd.DataFrame(columns=[
        "audio_path",
        "reference_raw",
        "prediction_raw",
        "audio_duration",
        "inference_time"
    ])

    processed = set()

In [ ]:
torch.cuda.empty_cache()

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()

start_time = time.time()

for idx in tqdm(range(len(df))):

    row = df.iloc[idx]
    audio_path = row["audio_path"]

    if audio_path in processed:
        continue

    try:

        info = sf.info(audio_path)
        duration = info.frames / info.samplerate

        audio, sr = librosa.load(audio_path, sr=16000)

        inputs = processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt"
        )

        input_values = inputs.input_values.to(device)

        t0 = time.time()

        with torch.no_grad():
            logits = model(input_values).logits

        pred_ids = torch.argmax(logits, dim=-1)
        pred_text = processor.batch_decode(pred_ids)[0]

        t1 = time.time()

        inference_time = t1 - t0

        new_row = {
            "audio_path": audio_path,
            "reference_raw": row["sentence"],
            "prediction_raw": pred_text,
            "audio_duration": duration,
            "inference_time": inference_time
        }

        results_df = pd.concat(
            [results_df, pd.DataFrame([new_row])],
            ignore_index=True
        )

        processed.add(audio_path)

        if len(results_df) % 50 == 0:
            results_df.to_csv(CHECKPOINT_PATH, index=False)
            print(f"Checkpoint saved: {len(results_df)}")

    except Exception as e:
        print("FAILED:", audio_path)
        print(e)

# final save
results_df.to_csv(CHECKPOINT_PATH, index=False)
print("Final checkpoint saved")

  0%|          | 0/10519 [00:00<?, ?it/s]/tmp/ipykernel_1966/1071912746.py:51: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat(
  0%|          | 52/10519 [00:04<16:03, 10.86it/s]

Checkpoint saved: 50


  1%|          | 102/10519 [00:08<15:43, 11.04it/s]

Checkpoint saved: 100


  1%|▏         | 152/10519 [00:12<12:14, 14.12it/s]

...

Checkpoint saved: 10450


100%|█████████▉| 10501/10519 [16:39<00:02,  8.34it/s]

Checkpoint saved: 10500


100%|██████████| 10519/10519 [16:40<00:00, 10.51it/s]


Final checkpoint saved


In [ ]:
import re
import pandas as pd
import hazm
from jiwer import wer, cer
import torch

CHECKPOINT_PATH = "/content/drive/MyDrive/asr_project/w2v_checkpoint.csv"

df = pd.read_csv(CHECKPOINT_PATH)

print("Loaded:", len(df))

In [ ]:
normalizer = hazm.Normalizer()

PERSIAN_CHAR_MAP = {
    "ك": "ک",
    "ي": "ی",
    "ى": "ی",
    "ؤ": "و",
    "ئ": "ی",
    "ة": "ه",
}

def normalize_persian(text):

    text = str(text)
    text = normalizer.normalize(text)

    for k, v in PERSIAN_CHAR_MAP.items():
        text = text.replace(k, v)

    text = re.sub(r"\d+", " <NUM> ", text)
    text = re.sub(r"[^\w\s<>]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
df["ref_norm"] = df["reference_raw"].apply(normalize_persian)
df["hyp_norm"] = df["prediction_raw"].apply(normalize_persian)

refs = df["ref_norm"].tolist()
hyps = df["hyp_norm"].tolist()

wer_score = wer(refs, hyps)
cer_score = cer(refs, hyps)

In [ ]:
total_audio = df["audio_duration"].sum()
total_inference = df["inference_time"].sum()

rtf = total_inference / total_audio
avg_latency = df["inference_time"].mean()

In [ ]:
if torch.cuda.is_available():
    peak_memory = torch.cuda.max_memory_allocated() / 1024**3
else:
    peak_memory = 0

In [ ]:
print("\n=========== WAV2VEC2 RESULTS ===========\n")
print("Samples:", len(df))
print("WER:", round(wer_score, 4))
print("CER:", round(cer_score, 4))
print("RTF:", round(rtf, 4))
print("Avg latency:", round(avg_latency, 4))
print("Peak GPU memory:", round(peak_memory, 2))
print("\n=======================================\n")

In [ ]:
summary = pd.DataFrame([{
    "model": "wav2vec2-large-xlsr-persian-v3",
    "samples": len(df),
    "wer": wer_score,
    "cer": cer_score,
    "rtf": rtf,
    "avg_latency_sec": avg_latency,
    "gpu_memory_gb": peak_memory,
    "total_inference_time_sec": total_inference
}])

summary.to_csv(
    "/content/drive/MyDrive/asr_project/w2v_summary.csv",
    index=False
)

print("Saved summary")

In [ ]:
pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 38.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re
import unicodedata
from jiwer import wer, cer

CHECKPOINT_PATH = "/content/drive/MyDrive/asr_project/w2v_persian_checkpoint.csv"

df = pd.read_csv(CHECKPOINT_PATH)

df = df.dropna(subset=["reference_raw", "prediction_raw"]).reset_index(drop=True)

In [ ]:
df.head()

,audio_path,reference_raw,prediction_raw,audio_duration,inference_time
0,/content/drive/MyDrive/asr_project/asr_data/fa...,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,2.753750,0.079157
1,/content/drive/MyDrive/asr_project/asr_data/fa...,دعا خوان,دعا خوان,3.103500,0.075789
2,/content/drive/MyDrive/asr_project/asr_data/fa...,اعتماد کرد,به اعتماد کرد,4.651043,0.055044
3,/content/drive/MyDrive/asr_project/asr_data/fa...,خب ، تو چیكار می كنی؟,خب تو چیکار می کنی,3.701625,0.049486
4,/content/drive/MyDrive/asr_project/asr_data/fa...,آه، نه اصلاُ!,آه نه اصلا,2.429625,0.037833


In [ ]:
df_sub = df[['prediction_raw', 'reference_raw']]
df_sub.head()

,prediction_raw,reference_raw
0,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم
1,دعا خوان,دعا خوان
2,به اعتماد کرد,اعتماد کرد
3,خب تو چیکار می کنی,خب ، تو چیكار می كنی؟
4,آه نه اصلا,آه، نه اصلاُ!


In [ ]:
pip install hazm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.9 MB/s eta 0:00:00
  Created wheel for flashtext: filename=flashtext-2.7-py2.py3-none-any.whl size=9300 sha256=e34be35af0d1ae1cec93ca8ec4c6827e6e7feab5e4c5a1071e2ba8c265b05e3f
  Stored in directory: /root/.cache/pip/wheels/8c/24/da/4d994d7a27cfc73a4e513a669fbeec4a71f871fe245a81977f
Successfully built flashtext


In [ ]:
import re
import unicodedata
import string
import hazm

_normalizer = hazm.Normalizer()

SKIP = set(["ā", "š", "="])

REPLACEMENTS = {
    "أ": "ا",
    "ك": "ک",
    "ي": "ی",
    "ى": "ی",
    "ﯽ": "ی",
    "ﻮ": "و",
    "ە": "ه",
    "ۀ": "ه",
}

DISCARD = set([
    "!", '"', "#", "&", "'", "(", ")", ",", "-", ".", ":",
    ";", "؟", "،", "؛", "ـ", "…", "«", "»", "–",
    "ً", "ٌ", "َ", "ُ", "ِ", "ّ", "ْ", "ٔ"
])

# remove ASCII digits + Persian/Arabic digits (consistent with HF eval)
DIGITS = set("0123456789۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩")


def w2v_persian_normalize(text: str) -> str:
    if text is None:
        return ""

    text = str(text)

    # 1. Unicode normalization (critical)
    text = unicodedata.normalize("NFKC", text)

    # 2. lower + strip
    text = text.lower().strip()

    # 3. Hazm normalization (Persian-specific standardization)
    text = _normalizer.normalize(text)

    # 4. remove hashtags (same as HF script)
    text = " ".join(w for w in text.split() if not w.startswith("#"))

    # 5. character replacements (Arabic → Persian canonical form)
    for k, v in REPLACEMENTS.items():
        text = text.replace(k, v)

    # 6. remove unwanted punctuation
    for tok in DISCARD:
        text = text.replace(tok, " ")

    # 7. remove SKIP characters (HF ignores these explicitly)
    for ch in SKIP:
        text = text.replace(ch, " ")

    # 8. remove digits (HF wav2vec2 eval does this implicitly via ignore list)
    text = "".join(ch for ch in text if ch not in DIGITS)

    # 9. remove Arabic letter variation "hamza"
    text = text.replace("ء", "")

    # 10. collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
df["ref_norm"] = df["reference_raw"].apply(w2v_persian_normalize)
df["hyp_norm"] = df["prediction_raw"].apply(w2v_persian_normalize)

In [ ]:
df.head()

,audio_path,reference_raw,prediction_raw,audio_duration,inference_time,ref_norm,hyp_norm
0,/content/drive/MyDrive/asr_project/asr_data/fa...,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,2.753750,0.079157,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم
1,/content/drive/MyDrive/asr_project/asr_data/fa...,دعا خوان,دعا خوان,3.103500,0.075789,دعا خوان,دعا خوان
2,/content/drive/MyDrive/asr_project/asr_data/fa...,اعتماد کرد,به اعتماد کرد,4.651043,0.055044,اعتماد کرد,به اعتماد کرد
3,/content/drive/MyDrive/asr_project/asr_data/fa...,خب ، تو چیكار می كنی؟,خب تو چیکار می کنی,3.701625,0.049486,خب تو چیکار می‌کنی,خب تو چیکار می‌کنی
4,/content/drive/MyDrive/asr_project/asr_data/fa...,آه، نه اصلاُ!,آه نه اصلا,2.429625,0.037833,آه نه اصلا,آه نه اصلا


### results

In [ ]:
wer_score = wer(df["ref_norm"].tolist(), df["hyp_norm"].tolist())
cer_score = cer(df["ref_norm"].tolist(), df["hyp_norm"].tolist())

print("NeMo-replicated WER:", wer_score)
print("NeMo-replicated CER:", cer_score)
print("Samples:", len(df))

NeMo-replicated WER: 0.26152512998266897
NeMo-replicated CER: 0.13370709666174596
Samples: 10517


In [ ]:
pip install kaldialign

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 3.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import kaldialign


def _alignment_error_rate(ref_tokens, hyp_tokens):
    """
    Compute error rate using Kaldi/Icefall alignment.

    Returns:
        error_rate, substitutions, deletions, insertions, reference_length
    """
    ali = kaldialign.align(ref_tokens, hyp_tokens, "*")

    subs = dels = ins = 0

    for r, h in ali:
        if r == "*":
            ins += 1
        elif h == "*":
            dels += 1
        elif r != h:
            subs += 1

    n_ref = len(ref_tokens)
    err = (subs + dels + ins) / max(1, n_ref)

    return err, subs, dels, ins, n_ref


def add_wer_cer_columns(
    df,
    ref_col="ref_norm",
    hyp_col="hyp_norm",
):
    """
    Adds per-sample WER and CER columns to a dataframe.

    Returns
    -------
    DataFrame
        Copy of dataframe with:
            wer
            cer
            wer_sub
            wer_del
            wer_ins
            cer_sub
            cer_del
            cer_ins
    """

    df = df.copy()

    wers = []
    cers = []

    wer_subs = []
    wer_dels = []
    wer_inss = []

    cer_subs = []
    cer_dels = []
    cer_inss = []

    for ref, hyp in zip(df[ref_col], df[hyp_col]):

        ref = "" if pd.isna(ref) else str(ref)
        hyp = "" if pd.isna(hyp) else str(hyp)

        # ---------- WER ----------
        wer, s, d, i, _ = _alignment_error_rate(
            ref.split(),
            hyp.split()
        )

        wers.append(wer)
        wer_subs.append(s)
        wer_dels.append(d)
        wer_inss.append(i)

        # ---------- CER ----------
        cer, s, d, i, _ = _alignment_error_rate(
            list(ref.replace(" ", "")),
            list(hyp.replace(" ", ""))
        )

        cers.append(cer)
        cer_subs.append(s)
        cer_dels.append(d)
        cer_inss.append(i)

    df["wer"] = wers
    df["cer"] = cers

    df["wer_sub"] = wer_subs
    df["wer_del"] = wer_dels
    df["wer_ins"] = wer_inss

    df["cer_sub"] = cer_subs
    df["cer_del"] = cer_dels
    df["cer_ins"] = cer_inss

    return df


def compute_dataset_wer_cer(
    df,
    ref_col="ref_norm",
    hyp_col="hyp_norm",
):
    """
    Computes overall dataset WER/CER (Icefall/Kaldi style).

    Returns
    -------
    dict
    """

    total_word_sub = total_word_del = total_word_ins = 0
    total_words = 0

    total_char_sub = total_char_del = total_char_ins = 0
    total_chars = 0

    for ref, hyp in zip(df[ref_col], df[hyp_col]):

        ref = "" if pd.isna(ref) else str(ref)
        hyp = "" if pd.isna(hyp) else str(hyp)

        _, s, d, i, n = _alignment_error_rate(
            ref.split(),
            hyp.split()
        )

        total_word_sub += s
        total_word_del += d
        total_word_ins += i
        total_words += n

        _, s, d, i, n = _alignment_error_rate(
            list(ref.replace(" ", "")),
            list(hyp.replace(" ", ""))
        )

        total_char_sub += s
        total_char_del += d
        total_char_ins += i
        total_chars += n

    wer = (
        total_word_sub + total_word_del + total_word_ins
    ) / max(1, total_words)

    cer = (
        total_char_sub + total_char_del + total_char_ins
    ) / max(1, total_chars)

    return {
        "WER": wer,
        "CER": cer,
        "word_sub": total_word_sub,
        "word_del": total_word_del,
        "word_ins": total_word_ins,
        "char_sub": total_char_sub,
        "char_del": total_char_del,
        "char_ins": total_char_ins,
        "total_words": total_words,
        "total_chars": total_chars,
    }

In [ ]:
# Add per-utterance metrics
df = add_wer_cer_columns(df)

# Inspect individual utterances
df[["hyp_norm", "ref_norm", "wer", "cer"]].head()

,hyp_norm,ref_norm,wer,cer
0,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,0.0,0.000000
1,دعا خوان,دعا خوان,0.0,0.000000
2,به اعتماد کرد,اعتماد کرد,0.5,0.222222
3,خب تو چیکار می‌کنی,خب تو چیکار می‌کنی,0.0,0.000000
4,آه نه اصلا,آه نه اصلا,0.0,0.000000


In [ ]:
df.head()

,audio_path,reference_raw,prediction_raw,audio_duration,inference_time,ref_norm,hyp_norm,wer,cer,wer_sub,wer_del,wer_ins,cer_sub,cer_del,cer_ins
0,/content/drive/MyDrive/asr_project/asr_data/fa...,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,2.753750,0.079157,از مهمونداری کنار بکشم,از مهمونداری کنار بکشم,0.0,0.000000,0,0,0,0,0,0
1,/content/drive/MyDrive/asr_project/asr_data/fa...,دعا خوان,دعا خوان,3.103500,0.075789,دعا خوان,دعا خوان,0.0,0.000000,0,0,0,0,0,0
2,/content/drive/MyDrive/asr_project/asr_data/fa...,اعتماد کرد,به اعتماد کرد,4.651043,0.055044,اعتماد کرد,به اعتماد کرد,0.5,0.222222,0,0,1,0,0,2
3,/content/drive/MyDrive/asr_project/asr_data/fa...,خب ، تو چیكار می كنی؟,خب تو چیکار می کنی,3.701625,0.049486,خب تو چیکار می‌کنی,خب تو چیکار می‌کنی,0.0,0.000000,0,0,0,0,0,0
4,/content/drive/MyDrive/asr_project/asr_data/fa...,آه، نه اصلاُ!,آه نه اصلا,2.429625,0.037833,آه نه اصلا,آه نه اصلا,0.0,0.000000,0,0,0,0,0,0


In [ ]:
def compute_dataset_wer_cer(
    df,
    ref_col="ref_norm",
    hyp_col="hyp_norm",
):
    """
    Computes overall dataset WER/CER (Icefall/Kaldi style).

    Returns
    -------
    dict
    """

    total_word_sub = total_word_del = total_word_ins = 0
    total_words = 0

    total_char_sub = total_char_del = total_char_ins = 0
    total_chars = 0

    for ref, hyp in zip(df[ref_col], df[hyp_col]):

        ref = "" if pd.isna(ref) else str(ref)
        hyp = "" if pd.isna(hyp) else str(hyp)

        _, s, d, i, n = _alignment_error_rate(
            ref.split(),
            hyp.split()
        )

        total_word_sub += s
        total_word_del += d
        total_word_ins += i
        total_words += n

        _, s, d, i, n = _alignment_error_rate(
            list(ref.replace(" ", "")),
            list(hyp.replace(" ", ""))
        )

        total_char_sub += s
        total_char_del += d
        total_char_ins += i
        total_chars += n

    wer = (
        total_word_sub + total_word_del + total_word_ins
    ) / max(1, total_words)

    cer = (
        total_char_sub + total_char_del + total_char_ins
    ) / max(1, total_chars)

    return {
        "WER": wer,
        "CER": cer,
        "word_sub": total_word_sub,
        "word_del": total_word_del,
        "word_ins": total_word_ins,
        "char_sub": total_char_sub,
        "char_del": total_char_del,
        "char_ins": total_char_ins,
        "total_words": total_words,
        "total_chars": total_chars,
    }

In [ ]:
results = compute_dataset_wer_cer(df)

print(f"WER: {results['WER']:.4f}")
print(f"CER: {results['CER']:.4f}")
print(results)

WER: 0.2615
CER: 0.1205
{'WER': 0.26152512998266897, 'CER': 0.12051892298513657, 'word_sub': 7852, 'word_del': 11808, 'word_ins': 1466, 'char_sub': 4495, 'char_del': 28584, 'char_ins': 4090, 'total_words': 80780, 'total_chars': 308408}


In [ ]:
df_sub_1 = df[df['cer'] > df['wer']]
df_sub_1

,audio_path,reference_raw,prediction_raw,audio_duration,inference_time,ref_norm,hyp_norm,wer,cer,wer_sub,wer_del,wer_ins,cer_sub,cer_del,cer_ins
21,/content/drive/MyDrive/asr_project/asr_data/fa...,هیچ گل مینا دارید؟,گوشی سمسنگ م هانزم تلویزیون علدی,10.517625,0.130065,هیچ گل مینا دارید,گوشی سمسنگ م هانزم تلویزیون علدی,1.500000,1.571429,4,0,2,9,0,13
108,/content/drive/MyDrive/asr_project/asr_data/fa...,(خنده) فقط، رفتن تو بزرگراه و گشت زدن,ها فقط رفتن تو بزرگراه و گشت زدن,8.789625,0.105831,خنده فقط رفتن تو بزرگراه و گشت زدن,ها فقط رفتن تو بزرگراه و گشت زدن,0.125000,0.148148,1,0,0,0,3,1
169,/content/drive/MyDrive/asr_project/asr_data/fa...,زبان مادری من فارسی است,زبان نیدامی من فارسی است,4.133625,0.056593,زبان مادری من فارسی است,زبان نیدامی من فارسی است,0.200000,0.210526,1,0,0,3,0,1
403,/content/drive/MyDrive/asr_project/asr_data/fa...,ال:روز زن چی؟,ال دو نقطه روز زن چی,7.949625,0.107258,ال روز زن چی,ال دو نقطه روز زن چی,0.500000,0.666667,0,0,2,0,0,6
747,/content/drive/MyDrive/asr_project/asr_data/fa...,تا بفهمیم چه كلاه بزرگى این جمهورى اسلامى سر م...,تا بفهمیم چه کلاه بزرگ بزرگی این جمهوری اسلامی...,9.701625,0.128513,تا بفهمیم چه کلاه بزرگی این جمهوری اسلامی سر م...,تا بفهمیم چه کلاه بزرگ بزرگی این جمهوری اسلامی...,0.062500,0.064516,0,0,1,0,0,4
1126,/content/drive/MyDrive/asr_project/asr_data/fa...,از یخچال جویی بر داریم,از یخچال جویی بر می بریم,3.701625,0.054067,از یخچال جویی بر داریم,از یخچال جویی بر می‌بریم,0.200000,0.222222,1,0,0,2,0,2
1302,/content/drive/MyDrive/asr_project/asr_data/fa...,تو زندگی ما فقط آب و هواس که تغییر میکنه,تو زندگی ما فقط آب و هواس که تره میکنه,5.741625,0.076226,تو زندگی ما فقط آب و هواس که تغییر میکنه,تو زندگی ما فقط آب و هواس که تره میکنه,0.100000,0.129032,1,0,0,0,3,1
1389,/content/drive/MyDrive/asr_project/asr_data/fa...,من مسیحی هستم.,من بثیر هستم,2.861625,0.049241,من مسیحی هستم,من بثیر هستم,0.333333,0.363636,1,0,0,3,1,0
1969,/content/drive/MyDrive/asr_project/asr_data/fa...,چندباری که زنگ درمون رو زده بود و اجازه دادیم ...,چندباری که زنگ درمون رو زده بود و اجازه دادیم ...,7.709625,0.107446,چندباری که زنگ درمون رو زده بود و اجازه دادیم ...,چندباری که زنگ درمون رو زده بود و اجازه دادیم ...,0.062500,0.071429,0,0,1,0,0,4
2106,/content/drive/MyDrive/asr_project/asr_data/fa...,حکم قطعی محکوم به او ابلاغ شد,خوب ان قطعی محکوم به او ابراق شد,4.327500,0.066565,حکم قطعی محکوم به او ابلاغ شد,خوب‌ان قطعی محکوم به او ابراق شد,0.285714,0.347826,2,0,0,5,0,3


# Whisper

In [ ]:
!pip -q install hazm jiwer

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 86.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re
import unicodedata
from jiwer import wer, cer

CHECKPOINT_PATH = (
    f"/content/drive/MyDrive/asr_project/"
    f"whisper_large-v3_checkpoint.csv"
)

df = pd.read_csv(CHECKPOINT_PATH)

df = df.dropna(subset=["reference_raw", "prediction_raw"]).reset_index(drop=True)
df.head()

,audio_path,reference_raw,prediction_raw,audio_duration,inference_time
0,/content/drive/MyDrive/asr_project/asr_data/fa...,از مهمونداری کنار بکشم,از مهمانداری کنار بکشم,2.753750,4.007099
1,/content/drive/MyDrive/asr_project/asr_data/fa...,دعا خوان,دعا خان,3.103500,0.619558
2,/content/drive/MyDrive/asr_project/asr_data/fa...,اعتماد کرد,اعتماد کرد,4.651043,0.684038
3,/content/drive/MyDrive/asr_project/asr_data/fa...,خب ، تو چیكار می كنی؟,خوبتو چیکارن کن,3.701625,1.076763
4,/content/drive/MyDrive/asr_project/asr_data/fa...,آه، نه اصلاُ!,نه اصلا,2.429625,0.847911


In [ ]:
df_sub = df[['prediction_raw', 'reference_raw']]
df_sub.head()

,prediction_raw,reference_raw
0,از مهمانداری کنار بکشم,از مهمونداری کنار بکشم
1,دعا خان,دعا خوان
2,اعتماد کرد,اعتماد کرد
3,خوبتو چیکارن کن,خب ، تو چیكار می كنی؟
4,نه اصلا,آه، نه اصلاُ!


In [ ]:
SKIP = set([
    "ā", "š", "="
])

REPLACEMENTS = {
    "أ": "ا",
    "ك": "ک",
    "ي": "ی",
    "ى": "ی",
    "ﯽ": "ی",
    "ﻮ": "و",
    "ە": "ه",
    "ۀ": "ه",
}

DISCARD = set([
    "!", '"', "#", "&", "'", "(", ")", ",", "-", ".", ":",
    ";", "؟", "،", "؛", "ـ", "…", "«", "»", "–",
    "ً", "ٌ", "َ", "ُ", "ِ", "ّ", "ْ", "ٔ"
])


def nemo_paper_normalize(text: str) -> str:
    if text is None:
        return ""

    text = str(text)

    # Unicode normalization FIRST (important)
    text = unicodedata.normalize("NFKC", text)

    # remove hashtags (NeMo-style filtering)
    text = " ".join(w for w in text.split() if not w.startswith("#"))

    # character replacements
    for k, v in REPLACEMENTS.items():
        text = text.replace(k, v)

    # remove punctuation tokens
    for tok in DISCARD:
        text = text.replace(tok, " ")

    # remove Arabic letter variations like hamza
    text = text.replace("ء", "")

    # collapse whitespace
    text = " ".join(text.split())

    return text

In [ ]:
df_sub["hyp_norm"] = df_sub["prediction_raw"].apply(nemo_paper_normalize)
df_sub["ref_norm"] = df_sub["reference_raw"].apply(nemo_paper_normalize)

/tmp/ipykernel_1188/1633250410.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sub["hyp_norm"] = df_sub["prediction_raw"].apply(nemo_paper_normalize)
/tmp/ipykernel_1188/1633250410.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sub["ref_norm"] = df_sub["reference_raw"].apply(nemo_paper_normalize)


In [ ]:
df_sub.loc[df_sub["hyp_norm"] == "", "hyp_norm"] = " "
df_sub.loc[df_sub["ref_norm"] == "", "ref_norm"] = " "

In [ ]:
wer_score = wer(df_sub["ref_norm"].tolist(), df_sub["hyp_norm"].tolist())
cer_score = cer(df_sub["ref_norm"].tolist(), df_sub["hyp_norm"].tolist())

print("NeMo-replicated WER:", wer_score)
print("NeMo-replicated CER:", cer_score)
print("Samples:", len(df_sub))

NeMo-replicated WER: 0.39246016426831376
NeMo-replicated CER: 0.2357303000747337
Samples: 10517


In [ ]:
pip install kaldialign

In [ ]:
import pandas as pd
import kaldialign


def _alignment_error_rate(ref_tokens, hyp_tokens):
    """
    Compute error rate using Kaldi/Icefall alignment.

    Returns:
        error_rate, substitutions, deletions, insertions, reference_length
    """
    ali = kaldialign.align(ref_tokens, hyp_tokens, "*")

    subs = dels = ins = 0

    for r, h in ali:
        if r == "*":
            ins += 1
        elif h == "*":
            dels += 1
        elif r != h:
            subs += 1

    n_ref = len(ref_tokens)
    err = (subs + dels + ins) / max(1, n_ref)

    return err, subs, dels, ins, n_ref


def add_wer_cer_columns(
    df,
    ref_col="ref_norm",
    hyp_col="hyp_norm",
):
    """
    Adds per-sample WER and CER columns to a dataframe.

    Returns
    -------
    DataFrame
        Copy of dataframe with:
            wer
            cer
            wer_sub
            wer_del
            wer_ins
            cer_sub
            cer_del
            cer_ins
    """

    df = df.copy()

    wers = []
    cers = []

    wer_subs = []
    wer_dels = []
    wer_inss = []

    cer_subs = []
    cer_dels = []
    cer_inss = []

    for ref, hyp in zip(df[ref_col], df[hyp_col]):

        ref = "" if pd.isna(ref) else str(ref)
        hyp = "" if pd.isna(hyp) else str(hyp)

        # ---------- WER ----------
        wer, s, d, i, _ = _alignment_error_rate(
            ref.split(),
            hyp.split()
        )

        wers.append(wer)
        wer_subs.append(s)
        wer_dels.append(d)
        wer_inss.append(i)

        # ---------- CER ----------
        cer, s, d, i, _ = _alignment_error_rate(
            list(ref.replace(" ", "")),
            list(hyp.replace(" ", ""))
        )

        cers.append(cer)
        cer_subs.append(s)
        cer_dels.append(d)
        cer_inss.append(i)

    df["wer"] = wers
    df["cer"] = cers

    df["wer_sub"] = wer_subs
    df["wer_del"] = wer_dels
    df["wer_ins"] = wer_inss

    df["cer_sub"] = cer_subs
    df["cer_del"] = cer_dels
    df["cer_ins"] = cer_inss

    return df


def compute_dataset_wer_cer(
    df,
    ref_col="ref_norm",
    hyp_col="hyp_norm",
):
    """
    Computes overall dataset WER/CER (Icefall/Kaldi style).

    Returns
    -------
    dict
    """

    total_word_sub = total_word_del = total_word_ins = 0
    total_words = 0

    total_char_sub = total_char_del = total_char_ins = 0
    total_chars = 0

    for ref, hyp in zip(df[ref_col], df[hyp_col]):

        ref = "" if pd.isna(ref) else str(ref)
        hyp = "" if pd.isna(hyp) else str(hyp)

        _, s, d, i, n = _alignment_error_rate(
            ref.split(),
            hyp.split()
        )

        total_word_sub += s
        total_word_del += d
        total_word_ins += i
        total_words += n

        _, s, d, i, n = _alignment_error_rate(
            list(ref.replace(" ", "")),
            list(hyp.replace(" ", ""))
        )

        total_char_sub += s
        total_char_del += d
        total_char_ins += i
        total_chars += n

    wer = (
        total_word_sub + total_word_del + total_word_ins
    ) / max(1, total_words)

    cer = (
        total_char_sub + total_char_del + total_char_ins
    ) / max(1, total_chars)

    return {
        "WER": wer,
        "CER": cer,
        "word_sub": total_word_sub,
        "word_del": total_word_del,
        "word_ins": total_word_ins,
        "char_sub": total_char_sub,
        "char_del": total_char_del,
        "char_ins": total_char_ins,
        "total_words": total_words,
        "total_chars": total_chars,
    }

In [ ]:
# Add per-utterance metrics
df_sub = add_wer_cer_columns(df_sub)

# Inspect individual utterances
df_sub[["hyp_norm", "ref_norm", "wer", "cer"]].head()

,hyp_norm,ref_norm,wer,cer
0,از مهمانداری کنار بکشم,از مهمونداری کنار بکشم,0.250000,0.052632
1,دعا خان,دعا خوان,0.500000,0.142857
2,اعتماد کرد,اعتماد کرد,0.000000,0.000000
3,خوبتو چیکارن کن,خب تو چیکار می کنی,1.000000,0.285714
4,نه اصلا,آه نه اصلا,0.333333,0.250000


In [ ]:
def compute_dataset_wer_cer(
    df,
    ref_col="ref_norm",
    hyp_col="hyp_norm",
):
    """
    Computes overall dataset WER/CER (Icefall/Kaldi style).

    Returns
    -------
    dict
    """

    total_word_sub = total_word_del = total_word_ins = 0
    total_words = 0

    total_char_sub = total_char_del = total_char_ins = 0
    total_chars = 0

    for ref, hyp in zip(df[ref_col], df[hyp_col]):

        ref = "" if pd.isna(ref) else str(ref)
        hyp = "" if pd.isna(hyp) else str(hyp)

        _, s, d, i, n = _alignment_error_rate(
            ref.split(),
            hyp.split()
        )

        total_word_sub += s
        total_word_del += d
        total_word_ins += i
        total_words += n

        _, s, d, i, n = _alignment_error_rate(
            list(ref.replace(" ", "")),
            list(hyp.replace(" ", ""))
        )

        total_char_sub += s
        total_char_del += d
        total_char_ins += i
        total_chars += n

    wer = (
        total_word_sub + total_word_del + total_word_ins
    ) / max(1, total_words)

    cer = (
        total_char_sub + total_char_del + total_char_ins
    ) / max(1, total_chars)

    return {
        "WER": wer,
        "CER": cer,
        "word_sub": total_word_sub,
        "word_del": total_word_del,
        "word_ins": total_word_ins,
        "char_sub": total_char_sub,
        "char_del": total_char_del,
        "char_ins": total_char_ins,
        "total_words": total_words,
        "total_chars": total_chars,
    }

compute_dataset_wer_cer(df_sub)

{'WER': 0.39246016426831376,
 'CER': 0.24328221221272767,
 'word_sub': 18522,
 'word_del': 5552,
 'word_ins': 5408,
 'char_sub': 12959,
 'char_del': 58545,
 'char_ins': 8811,
 'total_words': 75121,
 'total_chars': 330131}

### results

In [ ]:
results = compute_dataset_wer_cer(df_sub)

print(f"WER: {results['WER']:.4f}")
print(f"CER: {results['CER']:.4f}")
print(results)

WER: 0.3677
CER: 0.1064
{'WER': 0.36772021227951057, 'CER': 0.10642468992705639, 'word_sub': 18521, 'word_del': 2609, 'word_ins': 5408, 'char_sub': 12959, 'char_del': 7979, 'char_ins': 8811, 'total_words': 72169, 'total_chars': 279531}


In [ ]:
df_sub[df_sub["ref_norm"].str.match(r"^(?=.*\d)(?=.*[^\d]).+", na=False)]

,prediction_raw,reference_raw,hyp_norm,ref_norm,wer,cer,wer_sub,wer_del,wer_ins,cer_sub,cer_del,cer_ins
1700,هی این خانم ایده جالبی داره اگذار گوش کنیم,هی، این خانم ایده جالبی دارد بگذار گوش کنیم\t\...,هی این خانم ایده جالبی داره اگذار گوش کنیم,هی این خانم ایده جالبی دارد بگذار گوش کنیم 2 0...,0.99729,0.999328,1,2943,0,0,50566,0


In [ ]:
df_sub_1 = df_sub[df_sub['cer'] > df_sub['wer']]
df_sub_1

,prediction_raw,reference_raw,hyp_norm,ref_norm,wer,cer,wer_sub,wer_del,wer_ins,cer_sub,cer_del,cer_ins
205,جد و جاحت,جد و جهد,جد و جاحت,جد و جهد,0.333333,0.500000,1,0,0,2,0,1
341,اوه این بازی مهمی نیست,اوه این قضیه مهمی نیست,اوه این بازی مهمی نیست,اوه این قضیه مهمی نیست,0.200000,0.222222,1,0,0,2,1,1
403,حل دو نقطه روز زن چی؟,ال:روز زن چی؟,حل دو نقطه روز زن چی,ال روز زن چی,0.750000,0.777778,1,0,2,1,0,6
648,قدیمی ترین بنا در شمال توکیو 50 هزار سال قدمت...,قدیمی ترین بنا در شمال توکیو پنجاه هزار سال قد...,قدیمی ترین بنا در شمال توکیو 50 هزار سال قدمت ...,قدیمی ترین بنا در شمال توکیو پنجاه هزار سال قد...,0.090909,0.116279,1,0,0,2,3,0
714,278 تا,دویست و هفتاد و هشت تا,278 تا,دویست و هفتاد و هشت تا,0.833333,0.882353,1,4,0,3,12,0
951,مردم در اینجا با اینجا با اینجا با اینجا با ا...,و این آب فایده ای ندارد!,مردم در اینجا با اینجا با اینجا با اینجا با ای...,و این آب فایده ای ندارد,12.500000,13.944444,6,0,69,8,0,243
1109,این قبیل زارد,این قبیل اظهارات,این قبیل زارد,این قبیل اظهارات,0.333333,0.357143,1,0,0,2,3,0
1244,بخش ۱۲۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰...,لطفاً یک گزارش سرقت پر کنید.,بخش ۱۲۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰۰...,لطفا یک گزارش سرقت پر کنید,1.000000,5.380952,2,4,0,21,0,92
1700,هی این خانم ایده جالبی داره اگذار گوش کنیم,هی، این خانم ایده جالبی دارد بگذار گوش کنیم\t\...,هی این خانم ایده جالبی داره اگذار گوش کنیم,هی این خانم ایده جالبی دارد بگذار گوش کنیم 2 0...,0.997290,0.999328,1,2943,0,0,50566,0
1763,ههههههههههههههههههههههههههههههههههههههههههههه...,(خنده) میخواست یک پولی دربیاورد,هههههههههههههههههههههههههههههههههههههههههههههه...,خنده میخواست یک پولی دربیاورد,1.000000,8.880000,1,4,0,24,0,198


In [ ]:
df_sub = df_sub.drop(index=1700)
df_sub.count()

,0
prediction_raw,10516
reference_raw,10516
hyp_norm,10516
ref_norm,10516
wer,10516
cer,10516
wer_sub,10516
wer_del,10516
wer_ins,10516
cer_sub,10516
